In [ ]:
from __future__ import annotations

from pathlib import Path
import os, sys, json, time, argparse, hashlib, pathlib, re, base64, requests
from dotenv import load_dotenv, find_dotenv
from IPython.display import IFrame, display, HTML, JSON

import pandas as pd
import pymupdf as fitz
from langchain_text_splitters import SpacyTextSplitter, RecursiveCharacterTextSplitter

def find_project_root() -> Path:
    p = Path.cwd()
    markers = {".git", "pyproject.toml", ".env"}
    for up in [p, *p.parents]:
        if any((up / m).exists() for m in markers):
            return up
    return p

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

env_path = find_dotenv(filename=".env", usecwd=True) or str(PROJECT_ROOT / ".env")
print("Loaded .env from:", env_path)
load_dotenv(env_path, override=False)

from ingest.constants import SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, STORAGE_BUCKET, PDF_FILENAME
from ingest.chunk_import import fetch_pdf_from_storage, build_toc, chunk_sections, flatten
from ingest.embedding_import import generate_embeddings, validate_embeddings

In [ ]:
'''Supabase PDF Retrieval'''

def fetch_pdf_from_storage(
    supabase_url: str,
    service_role_key: str,
    bucket: str,
    filename: str
) -> bytes:
    """
    Fetch PDF from Supabase Storage using service_role key.
    Works for private buckets.
    """
    auth_url = f"{supabase_url}/storage/v1/object/authenticated/{bucket}/{filename}"
    
    headers = {
        "apikey": service_role_key,
        "Authorization": f"Bearer {service_role_key}"
    }
    
    print(f"📥 Fetching PDF...")
    resp = requests.get(auth_url, headers=headers, timeout=30)
    resp.raise_for_status()
    
    pdf_bytes = resp.content
    if not pdf_bytes.startswith(b'%PDF'):
        raise ValueError("Downloaded file is not a valid PDF")
    
    print(f"✅ Downloaded {len(pdf_bytes):,} bytes ({len(pdf_bytes) / 1024 / 1024:.2f} MB)")
    return pdf_bytes

# Fetch PDF
pdf_bytes = fetch_pdf_from_storage(
    SUPABASE_URL,
    SUPABASE_SERVICE_ROLE_KEY,
    STORAGE_BUCKET,
    PDF_FILENAME
)

# Generate doc key
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]
print(f"Doc key: {doc_key}")

# Display PDF directly from bytes using data URL
print("\n📄 Displaying PDF:")
pdf_base64 = base64.b64encode(pdf_bytes).decode('utf-8')
display(HTML(f'<iframe src="data:application/pdf;base64,{pdf_base64}" width="800" height="600"></iframe>'))

In [ ]:
'''ToC and Chunks Workflow'''

print("\n" + "=" * 60)
print("STEP 2: BUILD TOC")
print("=" * 60)
doc_key, toc, page_count = build_toc(pdf_bytes)

print(f"\n📚 ToC Summary:")
print(f"   Doc key: {doc_key}")
print(f"   Pages: {page_count}")
print(f"   Top-level sections: {len(toc)}")

# Flatten to see all nodes
flat_toc = flatten(toc)
print(f"   Total ToC nodes: {len(flat_toc)}")

# Show ToC structure
print(f"\n📖 ToC Structure (first 10 nodes):")
toc_preview = []
for node in flat_toc[10:20]:
    toc_preview.append({
        "id": node["id"],
        "level": node["level"],
        "title": node["title"][:50],
        "pages": f"{node['page_start']}-{node['page_end']}"
    })

df_toc = pd.DataFrame(toc_preview)
display(df_toc)

# Interactive JSON view of full ToC hierarchy
print(f"\n🌳 Full ToC Hierarchy (interactive):")
display(JSON(toc[:10]))  # Show first 3 chapters

# 3. Build Chunks
print("\n" + "=" * 60)
print("STEP 3: BUILD CHUNKS")
print("=" * 60)
chunks = chunk_sections(pdf_bytes, toc, chunk_size=1000, overlap=150)

print(f"\n✂️  Chunks Summary:")
print(f"   Total chunks: {len(chunks)}")
print(f"   Sections with chunks: {len(set(c['section_id'] for c in chunks))}")

# Chunk statistics
chunk_lengths = [len(c["text"]) for c in chunks]
print(f"   Avg chunk length: {sum(chunk_lengths) / len(chunk_lengths):.0f} chars")
print(f"   Min chunk length: {min(chunk_lengths)} chars")
print(f"   Max chunk length: {max(chunk_lengths)} chars")

# Show sample chunks
print(f"\n📄 Sample Chunks (first 5):")
chunks_preview = []
for c in chunks[:5]:
    chunks_preview.append({
        "section_id": c["section_id"][:30] + "...",
        "seq": c["chunk_seq"],
        "title": c["section_title"][:40],
        "level": c["level"],
        "pages": f"{c['page_start']}-{c['page_end']}",
        "chars": len(c["text"]),
        "text_preview": c["text"][:80] + "..."
    })

df_chunks = pd.DataFrame(chunks_preview)
display(df_chunks)

# Show chunks per section
print(f"\n📊 Chunks per Section (top 10):")
chunks_per_section = {}
for c in chunks:
    sid = c["section_id"]
    chunks_per_section[sid] = chunks_per_section.get(sid, 0) + 1

top_sections = sorted(chunks_per_section.items(), key=lambda x: x[1], reverse=True)[:10]
section_stats = []
for sid, count in top_sections:
    # Find section title
    title = next((c["section_title"] for c in chunks if c["section_id"] == sid), "Unknown")
    section_stats.append({
        "section_id": sid[:40],
        "title": title[:50],
        "chunk_count": count
    })

df_sections = pd.DataFrame(section_stats)
display(df_sections)

print(f"\n✅ ToC and Chunks built successfully!")
print(f"\n💡 Next step: Validate chunks quality before embedding")

In [ ]:
'''Embedding Ingestion'''

OUT_DIR = Path("data/out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Fetch PDF
pdf_bytes = fetch_pdf_from_storage(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY, STORAGE_BUCKET, PDF_FILENAME)
doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]

# 2. Build ToC
doc_key, toc, page_count = build_toc(pdf_bytes)
print(f"Built ToC: {page_count} pages")

# 3. Chunk sections
chunks = chunk_sections(pdf_bytes, toc, chunk_size=1000, overlap=150)
print(f"Generated {len(chunks)} chunks")

# 4. Generate embeddings
texts = [c["text"] for c in chunks]
embeddings = generate_embeddings(texts)

# 5. Validate
passed, issues = validate_embeddings(embeddings)
if not passed:
    print("⚠️ Embedding issues:", issues)
else:
    print("✅ All validations passed")

# 6. Add embeddings to chunks + doc_key
for i, c in enumerate(chunks):
    c["chunk_id"] = f"{doc_key}::{c['section_id']}::c{c['chunk_seq']:06d}"
    c["doc_key"] = doc_key
    c["embedding"] = embeddings[i].tolist()

# 7. Now push to Supabase (next cell)

In [ ]:
# Notebook: Test PDF Fetch from Supabase Storage
import hashlib
import requests
from pathlib import Path
import pymupdf as fitz

# Import from your constants file
from ingest.constants import (
    SUPABASE_URL,
    SUPABASE_SERVICE_ROLE_KEY,
    STORAGE_BUCKET,
    PDF_FILENAME,
    supabase  # Already initialized client
)

print("🔧 Configuration loaded:")
print(f"  SUPABASE_URL: {SUPABASE_URL}")
print(f"  Service key: {'✅ Set' if SUPABASE_SERVICE_ROLE_KEY else '❌ Missing'}")
print(f"  Bucket: {STORAGE_BUCKET}")
print(f"  Filename: {PDF_FILENAME}")
print()

def fetch_pdf_from_storage() -> bytes:
    """Fetch PDF using service_role key for maximum permissions"""
    
    # Method 1: Try public URL first (fastest if bucket is public)
    public_url = f"{SUPABASE_URL}/storage/v1/object/public/{STORAGE_BUCKET}/{PDF_FILENAME}"
    
    # Method 2: Authenticated endpoint (works for private buckets)
    auth_url = f"{SUPABASE_URL}/storage/v1/object/authenticated/{STORAGE_BUCKET}/{PDF_FILENAME}"
    
    headers = {
        "apikey": SUPABASE_SERVICE_ROLE_KEY,
        "Authorization": f"Bearer {SUPABASE_SERVICE_ROLE_KEY}"
    }
    
    print("📥 Fetching PDF...")
    
    # Try public first
    try:
        print(f"  Trying public URL...")
        resp = requests.get(public_url, headers=headers, timeout=30)
        
        if resp.status_code == 200:
            print(f"  ✅ Success via public URL")
            return resp.content
        else:
            print(f"  ⚠️  Public URL returned {resp.status_code}")
    except Exception as e:
        print(f"  ⚠️  Public URL failed: {e}")
    
    # Try authenticated
    try:
        print(f"  Trying authenticated URL...")
        resp = requests.get(auth_url, headers=headers, timeout=30)
        resp.raise_for_status()
        print(f"  ✅ Success via authenticated URL")
        return resp.content
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ HTTP {resp.status_code}: {resp.text}")
        raise RuntimeError(f"Failed to fetch PDF: {e}")
    except Exception as e:
        raise RuntimeError(f"Request failed: {e}")

# ============ RUN TEST ============

try:
    # Fetch PDF
    pdf_bytes = fetch_pdf_from_storage()
    
    # Validate it's a PDF
    if not pdf_bytes.startswith(b'%PDF'):
        raise ValueError(f"Invalid PDF file. First bytes: {pdf_bytes[:20]}")
    
    print(f"\n✅ Valid PDF downloaded!")
    print(f"   Size: {len(pdf_bytes):,} bytes ({len(pdf_bytes) / 1024 / 1024:.2f} MB)")
    
    # Generate document key
    doc_key = hashlib.sha256(pdf_bytes).hexdigest()[:16]
    print(f"   Doc key: {doc_key}")
    
    # Cache locally
    cache_dir = Path("data/cache")
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"{doc_key}.pdf"
    cache_path.write_bytes(pdf_bytes)
    print(f"   📁 Cached to: {cache_path}")
    
    # Open with PyMuPDF to verify + show metadata
    with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
        print(f"\n📄 PDF Metadata:")
        print(f"   Pages: {doc.page_count}")
        print(f"   Title: {doc.metadata.get('title', 'N/A')}")
        print(f"   Author: {doc.metadata.get('author', 'N/A')}")
        
        # Check if ToC exists
        toc = doc.get_toc(simple=True)
        print(f"   ToC entries: {len(toc) if toc else 0}")
        
        # Preview first page
        first_page_text = doc[0].get_text("text")
        print(f"\n📖 First page preview:")
        print(f"   {first_page_text[:250].strip()}...")
    
    print(f"\n🎉 SUCCESS! PDF fetch and validation complete.")
    
except Exception as e:
    print(f"\n❌ FAILED")
    print(f"   Error: {e}")
    print(f"\n🔍 Troubleshooting:")
    print(f"   1. Verify file exists in Supabase Storage Dashboard")
    print(f"   2. Check bucket name: '{STORAGE_BUCKET}'")
    print(f"   3. Check file path: '{PDF_FILENAME}'")
    print(f"   4. Ensure service_role key has storage permissions")